# MAR datasets Exploratory Data Analysis

This notebook performs Exploratory Data Analysis (EDA) on the MARS online learning dataset. The goal is to understand the distribution of users, courses, and interactions, and to identify any data quality issues before building the recommendation system.

## Import library & Configs

In [26]:
import os 

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


import warnings
warnings.filterwarnings("ignore")

In [27]:
dataset_dir = "/home/michael/Documents/Programs/thesis/mars-rec-sys/data/v2"
users_path = os.path.join(dataset_dir, "users.csv")
items_path = os.path.join(dataset_dir, "items.csv")
implicit_path = os.path.join(dataset_dir, "implicit_ratings.csv")
explicit_path = os.path.join(dataset_dir, "explicit_ratings.csv")


users    = pd.read_csv(users_path)
items    = pd.read_csv(items_path)
implicit = pd.read_csv(implicit_path)
explicit = pd.read_csv(explicit_path)

print("users   :", users.shape)
print("items   :", items.shape)
print("implicit:", implicit.shape)
print("explicit:", explicit.shape)

users   : (141387, 2)
items   : (2213, 12)
implicit: (462303, 3)
explicit: (153907, 5)


In [28]:
# Parse datetime
implicit["created_at"] = pd.to_datetime(
    implicit["created_at"],
    errors="coerce"
)

explicit["created_at"] = pd.to_datetime(
    explicit["created_at"],
    errors="coerce"
)

In [29]:
summary = {
    "Table": ["users", "items", "implicit_ratings", "explicit_ratings"],
    "Rows":  [len(users), len(items), len(implicit), len(explicit)],
    "Columns": [users.shape[1], items.shape[1], implicit.shape[1], explicit.shape[1]],
    "Null cells": [users.isnull().sum().sum(), items.isnull().sum().sum(),
                   implicit.isnull().sum().sum(), explicit.isnull().sum().sum()],
}
print(pd.DataFrame(summary).to_string(index=False))
print()

           Table   Rows  Columns  Null cells
           users 141387        2      123826
           items   2213       12        3012
implicit_ratings 462303        3           0
explicit_ratings 153907        5           0



## 1. Overview

Let's take a quick look at the first few rows of each table (users, items, implicit_ratings, explicit_ratings) to understand the columns and data formats.

In [30]:
for name, df in [("users", users), ("items", items), ("implicit", implicit), ("explicit", explicit)]:
    print(f"── {name} ──")
    display(df.head(3))


── users ──


,user_id,job
0,1,NaN
1,2,NaN
2,3,NaN


── items ──


,item_id,language,name,nb_views,description,created_at,Difficulty,Job,Software,Theme,duration,type
0,1,fr,Communiquer : qui peut m'aider ? Qui est dispo...,1611,<p>Découvrez comment améliorer votre communica...,2016,1 - Découverte,NaN,NaN,NaN,1980.0,webcast
1,2,fr,Formation Microsoft 365 - Le partage documentaire,1266,<p>Découvrez comment optimiser le partage de d...,2016,NaN,NaN,NaN,NaN,1757.0,webcast
2,3,fr,Optimiser la collaboration,756,Découvrez comment optimiser la collaboration p...,2016,1 - Découverte,NaN,NaN,NaN,2231.0,webcast


── implicit ──


,item_id,user_id,created_at
0,96,1,2017-02-02 16:01:59
1,100,1,2017-02-02 16:02:04
2,371,1,2017-04-18 22:15:29


── explicit ──


,user_id,item_id,watch_percentage,created_at,rating
0,4,324389,100,2022-12-15 23:00:59,10
1,9,323,75,2019-02-20 23:03:36,8
2,23,73683,12,2020-02-04 14:28:13,1


## 2. User Coverage & Cold-Start

Here we analyze the proportion of users present in both the `implicit` and `explicit` datasets. Notice that a large chunk of users (around 79%) have absolutely no interactions (cold-start users). This is a major challenge for building our recommendation model.

In [31]:
total_users    = len(users)
imp_users      = implicit["user_id"].nunique()
exp_users      = explicit["user_id"].nunique()
overlap_users  = len(set(implicit["user_id"]) & set(explicit["user_id"]))

labels  = ["Implicit only", "Explicit only", "Both", "No interaction (cold-start)"]
values  = [
    imp_users - overlap_users,
    exp_users - overlap_users,
    overlap_users,
    total_users - imp_users - exp_users + overlap_users,
]

fig = go.Figure(go.Pie(
    labels=labels, values=values,
    hole=0.45,
    marker_colors=["#4f98a3", "#e8af34", "#6daa45", "#c0c0c0"],
    textinfo="label+percent+value",
    textfont_size=12,
))
fig.update_layout(
    title="User Coverage Breakdown (Total: {:,})".format(total_users),
    width=700, height=480,
    showlegend=True,
)
fig.show()
print(f"Cold-start users: {values[3]:,} ({values[3]/total_users*100:.1f}%)")
print(f"Overlap (both):   {overlap_users:,}")


Cold-start users: 111,685 (79.0%)
Overlap (both):   15,419


As the chart shows, about 79% (over 111k) of users have no interactions across either dataset (pure cold-start). Only a small group (about 15.4k) appears in both the implicit and explicit datasets.

## 3. Interactions per User (Power-Law)

Let's check the distribution of interactions per user. This distribution typically follows a Power-Law (Long-tail) shape, meaning most users have very few interactions, while a tiny fraction of users are extremely active.

In [32]:
imp_counts = implicit.groupby("user_id").size()
exp_counts = explicit.groupby("user_id").size()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Implicit interactions / user", "Explicit ratings / user"])

for col_idx, (counts, label, color) in enumerate([
    (imp_counts, "Implicit", "#4f98a3"),
    (exp_counts, "Explicit", "#e8af34"),
], start=1):
    hist_vals = counts.clip(upper=200)  # cap for readability
    fig.add_trace(go.Histogram(
        x=hist_vals, nbinsx=60,
        marker_color=color, opacity=0.8,
        name=label,
    ), row=1, col=col_idx)
    fig.add_vline(x=counts.median(), line_dash="dash", line_color="red",
                  annotation_text=f"median={counts.median():.0f}",
                  row=1, col=col_idx)

fig.update_layout(
    title="Interactions per User (capped at 200 for readability)",
    height=420, width=950, showlegend=False,
    bargap=0.05,
)
fig.update_xaxes(title_text="# interactions")
fig.update_yaxes(title_text="# users")
fig.show()

for label, counts in [("Implicit", imp_counts), ("Explicit", exp_counts)]:
    print(f"{label}: mean={counts.mean():.1f}, median={counts.median():.0f}, max={counts.max()}, "
          f"users_with_1={( counts==1).sum()} ({(counts==1).mean()*100:.1f}%)")


Implicit: mean=15.7, median=4, max=4137, users_with_1=6417 (21.9%)
Explicit: mean=9.8, median=2, max=1094, users_with_1=5728 (36.4%)


The chart clearly highlights the long-tail nature of our data. For implicit data, the average user has around 15.7 interactions but the median is just 4, which means the average is skewed by a few highly active users (one user even has over 4000 interactions!). The explicit data tells a similar story, with about 36% of users only having a single rating.

## 4. Matrix Sparsity

Let's calculate the sparsity of the User-Item interaction matrix. A sparsity level above 99% is actually quite common in real-world recommendation systems.

In [33]:
n_users = users["user_id"].nunique()
n_items = items["item_id"].nunique()

imp_interactions = len(implicit)
exp_interactions = len(explicit)

imp_sparsity = 1 - imp_interactions / (n_users * n_items)
exp_sparsity = 1 - exp_interactions / (n_users * n_items)

fig = go.Figure()
for label, filled, color in [
    ("Implicit", (1 - imp_sparsity) * 100, "#4f98a3"),
    ("Explicit", (1 - exp_sparsity) * 100, "#e8af34"),
]:
    fig.add_trace(go.Bar(
        name=label,
        x=[label],
        y=[(1 - (imp_sparsity if label == "Implicit" else exp_sparsity)) * 100],
        marker_color=color,
        text=[f"{filled:.2f}% filled"],
        textposition="inside",
    ))
    fig.add_trace(go.Bar(
        name=f"{label} sparse",
        x=[label],
        y=[(imp_sparsity if label == "Implicit" else exp_sparsity) * 100],
        marker_color="#e0e0e0",
        text=[f"{(imp_sparsity if label == 'Implicit' else exp_sparsity)*100:.2f}% sparse"],
        textposition="inside",
        showlegend=False,
    ))

fig.update_layout(
    barmode="stack",
    title="User-Item Matrix Sparsity",
    yaxis_title="Percentage (%)",
    height=420, width=600,
)
fig.show()
print(f"Implicit sparsity: {imp_sparsity*100:.2f}%")
print(f"Explicit sparsity: {exp_sparsity*100:.2f}%")


Implicit sparsity: 99.85%
Explicit sparsity: 99.95%


The user-item matrix is extremely sparse (around 99.9% for both). With this level of sparsity, traditional Collaborative Filtering algorithms might struggle. We'll likely need to use Latent Factor Models or incorporate content-based features.

## 5. Explicit Rating Distribution

Let's look at how the explicit ratings are distributed. The data suffers from heavy positivity bias, with the vast majority being perfect scores (10).

In [34]:
rating_counts = explicit["rating"].value_counts().sort_index()

fig = go.Figure(go.Bar(
    x=rating_counts.index.astype(str),
    y=rating_counts.values,
    marker_color=["#c0c0c0"] * 9 + ["#e8111a"],  # highlight rating=10
    text=[f"{v:,}<br>({v/len(explicit)*100:.1f}%)" for v in rating_counts.values],
    textposition="outside",
))
fig.update_layout(
    title="Explicit Rating Distribution (Positivity Bias!)",
    xaxis_title="Rating",
    yaxis_title="Count",
    height=450, width=800,
)
fig.show()
print(f"Rating = 10: {(explicit['rating']==10).sum():,} ({(explicit['rating']==10).mean()*100:.1f}%)")


Rating = 10: 116,005 (75.4%)


The ratings are highly imbalanced. Over 75.4% of the ratings are perfect 10s. This positivity bias suggests that users only bother rating content they really love, or maybe the UI defaults to a high score. Because of this, it might make more sense to treat rating prediction as a binary classification problem rather than direct regression.

## 6. Watch Percentage Distribution

Let's check the watch percentage distribution. Most users finish the videos (100%), but there's some noisy data where the `watch_percentage` exceeds 100%.

In [35]:
if "watch_percentage" in explicit.columns:
    wp = explicit["watch_percentage"].dropna()
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=wp.clip(0, 110), nbinsx=55,
        marker_color="#4f98a3", opacity=0.8,
        name="watch %",
    ))
    fig.add_vline(x=wp.median(), line_dash="dash", line_color="red",
                  annotation_text=f"median={wp.median():.0f}%")
    fig.add_vline(x=wp.mean(),   line_dash="dot",  line_color="orange",
                  annotation_text=f"mean={wp.mean():.1f}%")
    fig.update_layout(
        title="Watch Percentage (capped at 110%)",
        xaxis_title="Watch %", yaxis_title="Count",
        height=420, width=800,
    )
    fig.show()
    print(f"watch_percentage: mean={wp.mean():.1f}%, median={wp.median():.0f}%, max={wp.max():.0f}%")
    print(f"Values > 100%: {(wp > 100).sum()} ({(wp > 100).mean()*100:.1f}%)")
else:
    print("Column 'watch_percentage' not found in explicit_ratings.csv")


watch_percentage: mean=87.0%, median=100%, max=198%
Values > 100%: 5477 (3.6%)


The average watch percentage is quite high (87%) with a median of 100%, indicating users usually watch the whole thing (or at least skip to the end). Interestingly, over 3.6% of the records show a watch percentage greater than 100% (some even reaching 198%).

## 7. Temporal Trends (Monthly)

Let's see how interactions and ratings fluctuate month by month to spot any platform activity trends.

In [36]:
imp_monthly = (implicit.set_index("created_at")
               .resample("ME")["user_id"].count().reset_index()
               .rename(columns={"created_at": "month", "user_id": "count"}))
exp_monthly = (explicit.set_index("created_at")
               .resample("ME")["user_id"].count().reset_index()
               .rename(columns={"created_at": "month", "user_id": "count"}))

fig = make_subplots(rows=2, cols=1,
    subplot_titles=["Implicit Interactions per Month", "Explicit Ratings per Month"],
    shared_xaxes=True)

fig.add_trace(go.Scatter(
    x=imp_monthly["month"], y=imp_monthly["count"],
    mode="lines+markers", name="Implicit",
    line=dict(color="#4f98a3", width=2),
    fill="tozeroy", fillcolor="rgba(79,152,163,0.15)",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=exp_monthly["month"], y=exp_monthly["count"],
    mode="lines+markers", name="Explicit",
    line=dict(color="#e8af34", width=2),
    fill="tozeroy", fillcolor="rgba(232,175,52,0.15)",
), row=2, col=1)

fig.update_layout(
    title="Monthly Activity Trend",
    height=600, width=950,
    showlegend=True,
)
fig.show()


The timeline chart shows that interactions were not completely stable, as they fluctuated from month to month. Several noticeable peaks can be observed, likely reflecting changes in learning behavior and external events. 

In particular, the highest number of ratings occurred in March 2020, which may have been strongly influenced by the COVID-19 pandemic, when many students shifted to online learning during lockdown periods.

## 8. Item Catalog Analysis

Let's analyze the item (course) attributes like type, duration, and check for missing metadata.

In [37]:
# Item type distribution
if "type" in items.columns:
    type_counts = items["type"].value_counts()
    fig = go.Figure(go.Pie(
        labels=type_counts.index, values=type_counts.values,
        hole=0.4,
        marker_colors=["#4f98a3", "#e8af34", "#6daa45", "#dd6974", "#a86fdf"],
        textinfo="label+percent+value",
    ))
    fig.update_layout(title="Item Type Distribution", height=420, width=650)
    fig.show()

# Missing metadata
missing = {
    "duration": items["duration"].isnull().sum() if "duration" in items.columns else 0,
    "difficulty": items["difficulty"].isnull().sum() if "difficulty" in items.columns else 0,
    "job_tag": (items["job"].isnull().sum() if "job" in items.columns
                else items.filter(like="job").isnull().all(axis=1).sum()),
}
print("Missing metadata:")
for k, v in missing.items():
    print(f"  {k}: {v:,} / {len(items):,} ({v/len(items)*100:.1f}%)")


Missing metadata:
  duration: 47 / 2,213 (2.1%)
  difficulty: 0 / 2,213 (0.0%)
  job_tag: 2,213 / 2,213 (100.0%)


In [38]:
if "duration" in items.columns:
    dur = items["duration"].dropna()
    fig = go.Figure(go.Histogram(
        x=dur.clip(upper=dur.quantile(0.99)),
        nbinsx=60,
        marker_color="#4f98a3", opacity=0.8,
    ))
    fig.add_vline(x=dur.median(), line_dash="dash", line_color="red",
                  annotation_text=f"median={dur.median():.0f}s")
    fig.add_vline(x=dur.mean(),   line_dash="dot",  line_color="orange",
                  annotation_text=f"mean={dur.mean():.0f}s")
    fig.update_layout(
        title="Item Duration Distribution (clipped at 99th pct)",
        xaxis_title="Duration (seconds)", yaxis_title="Count",
        height=400, width=800,
    )
    fig.show()
    print(f"Duration: mean={dur.mean():.0f}s, median={dur.median():.0f}s, max={dur.max():.0f}s")


Duration: mean=297s, median=133s, max=5900s


Most videos and lessons are pretty short, averaging around 5 minutes (297s) with a median of just over 2 minutes (133s). However, there are a few extreme outliers that drag on for nearly 1.5 hours (5900s).

## 9. Item Popularity & Long-Tail

Let's explore item popularity. Similar to users, item popularity also follows a long-tail distribution—a few items get all the attention while the rest get very little.

In [39]:
item_pop_imp = implicit.groupby("item_id").size().sort_values(ascending=False).reset_index()
item_pop_imp.columns = ["item_id", "interactions"]
item_pop_imp["rank"] = range(1, len(item_pop_imp) + 1)

item_pop_exp = explicit.groupby("item_id").size().sort_values(ascending=False).reset_index()
item_pop_exp.columns = ["item_id", "ratings"]
item_pop_exp["rank"] = range(1, len(item_pop_exp) + 1)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Implicit – Item Popularity (log scale)", "Explicit – Item Popularity (log scale)"])

fig.add_trace(go.Scatter(
    x=item_pop_imp["rank"], y=item_pop_imp["interactions"],
    mode="lines", line=dict(color="#4f98a3", width=2), name="Implicit",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=item_pop_exp["rank"], y=item_pop_exp["ratings"],
    mode="lines", line=dict(color="#e8af34", width=2), name="Explicit",
), row=1, col=2)

fig.update_xaxes(type="log", title_text="Item rank (log)")
fig.update_yaxes(type="log", title_text="# interactions (log)")
fig.update_layout(title="Long-Tail Item Popularity", height=420, width=950)
fig.show()

print("Implicit – top item:", item_pop_imp.iloc[0]["interactions"], "interactions")
print("Implicit – median  :", item_pop_imp["interactions"].median())
print("Explicit – top item:", item_pop_exp.iloc[0]["ratings"], "ratings")
print("Explicit – median  :", item_pop_exp["ratings"].median())


Implicit – top item: 6645 interactions
Implicit – median  : 83.0
Explicit – top item: 1891 ratings
Explicit – median  : 40.0


Just like the users, item popularity is heavily skewed in a long-tail distribution. The most popular items pull in thousands of interactions (peaking at over 6.6k), while the median is only around 40-80. We need to be careful of Popularity Bias when building our models.

## 10. User Demographics (Job)

Let's look at the users' job roles. A lot of this demographic data is missing, which will limit its usefulness for content-based filtering.

In [40]:
if "job" in users.columns:
    job_missing = users["job"].isnull().mean() * 100
    print(f"Users with job info: {100 - job_missing:.1f}%  |  Missing: {job_missing:.1f}%")

    top_jobs = (users["job"].value_counts().head(15))
    fig = go.Figure(go.Bar(
        x=top_jobs.values[::-1],
        y=top_jobs.index[::-1],
        orientation="h",
        marker_color="#4f98a3",
        text=top_jobs.values[::-1],
        textposition="outside",
    ))
    fig.update_layout(
        title="Top 15 User Jobs (among users with job data)",
        xaxis_title="Count", yaxis_title="Job",
        height=520, width=800,
        margin=dict(l=200),
    )
    fig.show()
else:
    print("Column 'job' not found in users.csv")


Users with job info: 12.4%  |  Missing: 87.6%


The user job information is mostly missing (87.6%). Trying to use this feature directly for predictions might just add noise or not improve accuracy much at all.

## 11. Data Quality & Recommendation Issues

Let's summarize the data noise and errors we've found, including sparsity, cold-start issues, rating bias, and nonsensical values (like watch_percentage > 100%).

In [41]:
imp_item_ids  = set(implicit["item_id"].unique())
cat_item_ids  = set(items["item_id"].unique())
orphan_items  = imp_item_ids - cat_item_ids

print("=" * 55)
print("DATA QUALITY SUMMARY")
print("=" * 55)
print(f"Cold-start users (no history)   : {total_users - imp_users:,} ({(total_users - imp_users)/total_users*100:.1f}%)")
print(f"Implicit matrix sparsity        : {imp_sparsity*100:.2f}%")
print(f"Explicit matrix sparsity        : {exp_sparsity*100:.2f}%")
print(f"Rating = 10 (positivity bias)   : {(explicit['rating']==10).sum():,} ({(explicit['rating']==10).mean()*100:.1f}%)")
print(f"Orphan item_ids in implicit     : {len(orphan_items):,}")
if "duration" in items.columns:
    print(f"Items missing duration          : {items['duration'].isnull().sum():,}")
if "difficulty" in items.columns:
    print(f"Items missing difficulty        : {items['difficulty'].isnull().sum():,}")
if "job" in users.columns:
    print(f"Users missing job               : {users['job'].isnull().sum():,} ({users['job'].isnull().mean()*100:.1f}%)")
if "watch_percentage" in explicit.columns:
    wp = explicit["watch_percentage"].dropna()
    print(f"watch_percentage > 100%         : {(wp>100).sum():,}")


DATA QUALITY SUMMARY
Cold-start users (no history)   : 112,020 (79.2%)
Implicit matrix sparsity        : 99.85%
Explicit matrix sparsity        : 99.95%
Rating = 10 (positivity bias)   : 116,005 (75.4%)
Orphan item_ids in implicit     : 334
Items missing duration          : 47
Users missing job               : 123,826 (87.6%)
watch_percentage > 100%         : 5,477


In [42]:
fig = px.scatter(
    explicit.sample(5000),  # sample cho đỡ lag
    x="watch_percentage",
    y="rating",
    trendline="ols",
    opacity=0.3,
    height=500,
    width=850,
)

fig.update_layout(
    yaxis=dict(range=[0,10])
)

fig.show()

The correlation between `watch_percentage` and `rating` is incredibly high (almost 0.997). Looking at the scatter plot and heatmap, there's a clear linear trend: the more someone watches, the higher their rating. It's highly likely that `rating` and `watch_percentage` aren't independent, but rather calculated by some underlying system rule rather than manual user input.

## 12. Data Cleaning: Handling Outliers

As we noticed earlier, there's a large chunk of records with `watch_percentage` > 100, and potentially some `rating` values exceeding the maximum of 10. This could be due to logging errors or users re-watching videos.

To ensure our data is clean before feeding it to the recommendation model, we'll clip these values:
- `watch_percentage`: capped at 100.
- `rating`: capped at 10.

In [43]:
# Prune watch_percentage > 100 về 100
if "watch_percentage" in explicit.columns:
    explicit['watch_percentage'] = explicit['watch_percentage'].clip(upper=100)

# Prune rating > 10 về 10
if "rating" in explicit.columns:
    explicit['rating'] = explicit['rating'].clip(upper=10)

print("Đã xử lý xong các giá trị bất thường!")
print(f"Max watch_percentage sau khi prune: {explicit['watch_percentage'].max():.0f}%")
print(f"Max rating sau khi prune: {explicit['rating'].max():.0f}")

# Hiển thị lại distribution một lần nữa sau khi prune để kiểm chứng
explicit[['watch_percentage', 'rating']].describe()

Đã xử lý xong các giá trị bất thường!
Max watch_percentage sau khi prune: 100%
Max rating sau khi prune: 10


,watch_percentage,rating
count,153907.000000,153907.000000
mean,86.976102,8.742331
std,27.419518,2.666145
min,0.000000,1.000000
25%,95.000000,10.000000
50%,100.000000,10.000000
75%,100.000000,10.000000
max,100.000000,10.000000


In [44]:
import os

# Create the directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save the processed dataframes to CSV
users.to_csv("../data/processed/users.csv", index=False)
items.to_csv("../data/processed/items.csv", index=False)
implicit.to_csv("../data/processed/implicit.csv", index=False)
explicit.to_csv("../data/processed/explicit.csv", index=False)

print("Data successfully saved to data/processed/")

Data successfully saved to data/processed/
